In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parents[2]

print("Project root:", project_root)
# Load the saved student assignments
split_path = project_root  / "data" / "processed" / "split_assignments.csv"

split_assignments = pd.read_csv(split_path)

# Keep only training-pool students
train_assignments = split_assignments[
    split_assignments["partition"] == "train_pool"
].copy()

# Convert fold numbers to integers
train_assignments["validation_fold"] = (
    train_assignments["validation_fold"].astype(int)
)

# Extract training student IDs
train_student_ids = set(train_assignments["user_id"])

print("Training students:", len(train_student_ids))
print("\nStudents assigned to each validation fold:")
print(train_assignments["validation_fold"].value_counts().sort_index())

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Training students: 3373

Students assigned to each validation fold:
validation_fold
1    675
2    675
3    675
4    674
5    674
Name: count, dtype: int64


In [3]:
data_path = (
    project_root
    / "data"
    / "processed"
    / "main_questions_cleaned.csv"
)

train_chunks = []

for chunk in pd.read_csv(data_path, chunksize=50000):

    training_rows = chunk[
        chunk["user_id"].isin(train_student_ids)
    ]

    train_chunks.append(training_rows)

# Combine all retained training interactions
train_df = pd.concat(train_chunks, ignore_index=True)

# Sort interactions chronologically within each student
train_df = (
    train_df.sort_values(["user_id", "order_id"])
            .reset_index(drop=True)
)

print("Training data shape:", train_df.shape)
print("Unique training students:", train_df["user_id"].nunique())
print("Missing values:\n", train_df.isnull().sum())

Training data shape: (219983, 7)
Unique training students: 3373
Missing values:
 user_id              0
order_id             0
problem_id           0
correct              0
skill_id         13449
skill_set            0
attempt_count        0
dtype: int64


In [4]:
# Check whether order_id is increasing within each student
is_chronological = (
    train_df.groupby("user_id")["order_id"]
    .apply(lambda s: s.is_monotonic_increasing)
)

print("All students chronologically sorted:", is_chronological.all())

# Count students whose order_id is not monotonic
print(
    "Students with incorrect ordering:",
    (~is_chronological).sum()
)

# Check duplicate order_id values within each student
duplicate_count = train_df.duplicated(
    subset=["user_id", "order_id"]
).sum()

print("Duplicate student-order pairs:", duplicate_count)

All students chronologically sorted: True
Students with incorrect ordering: 0
Duplicate student-order pairs: 0


In [5]:
fold_datasets = []

for fold_number in range(1, 6):

    # Students belonging to this validation fold
    val_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] == fold_number,
        "user_id"
    ]

    # All remaining training-pool students
    fold_train_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] != fold_number,
        "user_id"
    ]

    # Construct training and validation DataFrames
    fold_train_df = train_df[
        train_df["user_id"].isin(fold_train_student_ids)
    ].copy()

    fold_val_df = train_df[
        train_df["user_id"].isin(val_student_ids)
    ].copy()

    fold_datasets.append((fold_train_df, fold_val_df))

    print(
        f"Fold {fold_number}: "
        f"Training rows = {len(fold_train_df)}, "
        f"Validation rows = {len(fold_val_df)}"
    )

Fold 1: Training rows = 177324, Validation rows = 42659
Fold 2: Training rows = 176273, Validation rows = 43710
Fold 3: Training rows = 170186, Validation rows = 49797
Fold 4: Training rows = 178311, Validation rows = 41672
Fold 5: Training rows = 177838, Validation rows = 42145


In [6]:
for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    train_students = set(fold_train_df["user_id"])
    val_students = set(fold_val_df["user_id"])

    overlap = train_students.intersection(val_students)

    print(
        f"Fold {fold_number}: "
        f"Student overlap = {len(overlap)}"
    )

Fold 1: Student overlap = 0
Fold 2: Student overlap = 0
Fold 3: Student overlap = 0
Fold 4: Student overlap = 0
Fold 5: Student overlap = 0


In [7]:
def add_overall_history_accuracy(data, p0, alpha=10, training=False):

    df = data.copy()

    # 1. Sort interactions chronologically for each student
    df = df.sort_values(
        ["user_id", "order_id"]
    ).copy()

    # 2. Count previous interactions for each student
    df["previous_interactions"] = (
        df.groupby("user_id").cumcount()
    )

    # 3. Count previous correct interactions
    df["previous_correct"] = (
        df.groupby("user_id")["correct"]
          .transform(
              lambda s: s.cumsum().shift(1, fill_value=0)
          )
    )

    # 4. Calculate the appropriate global prior
    if training:
        total_correct = df["correct"].sum()
        total_interactions = len(df)

        row_prior = (
            total_correct - df["correct"]
        ) / (total_interactions - 1)

    else:
        row_prior = p0

    # 5. Calculate smoothed historical accuracy
    df["overall_smoothed_accuracy"] = (
        df["previous_correct"] + alpha * row_prior
    ) / (
        df["previous_interactions"] + alpha
    )

    return df

In [8]:
v11_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    # Global correct rate from this fold's training data
    p0 = fold_train_df["correct"].mean()

    # Construct training features
    v11_train_df = add_overall_history_accuracy(
        fold_train_df,
        p0,
        alpha=10,
        training=True
    )

    # Construct validation features
    v11_val_df = add_overall_history_accuracy(
        fold_val_df,
        p0,
        alpha=10,
        training=False
    )

    v11_fold_datasets.append(
        (v11_train_df, v11_val_df)
    )

    print(
        f"Fold {fold_number}: "
        f"Training shape = {v11_train_df.shape}, "
        f"Validation shape = {v11_val_df.shape}"
    )

Fold 1: Training shape = (177324, 10), Validation shape = (42659, 10)
Fold 2: Training shape = (176273, 10), Validation shape = (43710, 10)
Fold 3: Training shape = (170186, 10), Validation shape = (49797, 10)
Fold 4: Training shape = (178311, 10), Validation shape = (41672, 10)
Fold 5: Training shape = (177838, 10), Validation shape = (42145, 10)


In [9]:
# Get Fold 1 training features
v11_train_df, v11_val_df = v11_fold_datasets[0]

# Find a student with at least five interactions
student_counts = v11_train_df["user_id"].value_counts()

sample_student_id = student_counts[
    student_counts >= 5
].index[0]

# Extract this student's first five interactions
student_history = (
    v11_train_df[
        v11_train_df["user_id"] == sample_student_id
    ]
    .head(5)
)

# Display relevant columns
print(
    student_history[
        [
            "user_id",
            "order_id",
            "correct",
            "previous_interactions",
            "previous_correct",
            "overall_smoothed_accuracy"
        ]
    ].to_string(index=False)
)

 user_id  order_id  correct  previous_interactions  previous_correct  overall_smoothed_accuracy
   78970  20765556        1                      0                 0                   0.660134
   78970  20765796        0                      1                 1                   0.691036
   78970  20766021        0                      2                 1                   0.633450
   78970  20766237        0                      3                 1                   0.584723
   78970  20855602        1                      4                 1                   0.542953


In [10]:
# Calculate the Fold 1 training prior
p0 = fold_train_df["correct"].mean()

# Leave-one-out prior for each training row
total_correct = fold_train_df["correct"].sum()
total_interactions = len(fold_train_df)

sample = student_history.copy()

sample["manual_accuracy"] = (
    sample["previous_correct"]
    + 10 * (
        total_correct - sample["correct"]
    ) / (total_interactions - 1)
) / (
    sample["previous_interactions"] + 10
)

print(
    sample[
        [
            "correct",
            "overall_smoothed_accuracy",
            "manual_accuracy"
        ]
    ].to_string(index=False)
)

print(
    "\nMaximum difference:",
    (
        sample["overall_smoothed_accuracy"]
        - sample["manual_accuracy"]
    ).abs().max()
)

 correct  overall_smoothed_accuracy  manual_accuracy
       1                   0.660134         0.657546
       0                   0.691036         0.688683
       0                   0.633450         0.631293
       0                   0.584723         0.582732
       1                   0.542953         0.541104

Maximum difference: 0.002588376132097503


In [11]:
# Explicitly retrieve Fold 1 training data
fold_train_df, fold_val_df = fold_datasets[0]

# Recalculate Fold 1 training statistics
total_correct = fold_train_df["correct"].sum()
total_interactions = len(fold_train_df)

# Recalculate manually for the selected student
sample = student_history.copy()

sample["manual_accuracy"] = (
    sample["previous_correct"]
    + 10 * (
        total_correct - sample["correct"]
    ) / (total_interactions - 1)
) / (
    sample["previous_interactions"] + 10
)

print(
    sample[
        [
            "correct",
            "overall_smoothed_accuracy",
            "manual_accuracy"
        ]
    ].to_string(index=False)
)

print(
    "\nMaximum difference:",
    (
        sample["overall_smoothed_accuracy"]
        - sample["manual_accuracy"]
    ).abs().max()
)

 correct  overall_smoothed_accuracy  manual_accuracy
       1                   0.660134         0.660134
       0                   0.691036         0.691036
       0                   0.633450         0.633450
       0                   0.584723         0.584723
       1                   0.542953         0.542953

Maximum difference: 0.0


In [12]:
feature_columns = ["overall_smoothed_accuracy"]

for fold_number, (train_data, val_data) in enumerate(
    v11_fold_datasets, start=1
):
    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    print(
        f"Fold {fold_number}: "
        f"X_train = {X_train.shape}, "
        f"X_val = {X_val.shape}"
    )

Fold 1: X_train = (177324, 1), X_val = (42659, 1)
Fold 2: X_train = (176273, 1), X_val = (43710, 1)
Fold 3: X_train = (170186, 1), X_val = (49797, 1)
Fold 4: X_train = (178311, 1), X_val = (41672, 1)
Fold 5: X_train = (177838, 1), X_val = (42145, 1)


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

v11_models = []
v11_results = []

for fold_number, (train_data, val_data) in enumerate(
    v11_fold_datasets, start=1
):

    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    # Create preprocessing and model pipeline
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    # Fit scaler and logistic regression using training data only
    model.fit(X_train, y_train)

    # Predict probability of correct outcome
    y_pred = model.predict_proba(X_val)[:, 1]

    # Evaluate validation predictions
    ll = log_loss(y_val, y_pred, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred)

    v11_models.append(model)

    v11_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}"
    )

Fold 1: Log Loss = 0.5948, Brier Score = 0.2037
Fold 2: Log Loss = 0.5898, Brier Score = 0.2012
Fold 3: Log Loss = 0.6045, Brier Score = 0.2081
Fold 4: Log Loss = 0.5949, Brier Score = 0.2037
Fold 5: Log Loss = 0.5860, Brier Score = 0.1997
